In [ ]:
!pip install missingno
!pip install xgboost lightgbm scikit-learn

In [ ]:
import os
import pandas as pd
import numpy as np
import warnings
import gc
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor)
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/anhduy54/43-temp/data_43_temp.csv", skiprows=[1,2]) #trước mắt bỏ 2 dòng đầu cho dễ xử lí
# df = pd.read_csv("/kaggle/input/datasets/anhduy54/43-wind/Data_WindSpeed_43.csv")
print(f"temp_dataset: {df.shape}")

---
# **Create Missing Data**

In [ ]:
def generate_fixed_length_gaps(df_input, gap_days, num_gaps=10, seed=42, rows_per_day=8):
    """
    Tạo bộ dữ liệu missing với CÙNG MỘT độ dài gap cho tất cả các vị trí được chọn.
    Có tích hợp thuật toán chống đè (anti-overlap) để đảm bảo các gap không dính vào nhau.
    """
    df_temp = df_input.copy(deep=True)
    feature_columns = [col for col in df_temp.columns if col != "TimeVN"]
    
    # Đảm bảo format datetime để tìm đúng 1:00 AM
    ts = pd.to_datetime(df_temp['TimeVN'], format='mixed', dayfirst=True, errors='coerce')
    
    # Lấy ra vị trí (integer index) của các dòng có giờ = 1
    # Dùng np.where để lấy vị trí tuyệt đối, an toàn hơn loc khi thao tác
    daily_start_positions = np.where(ts.dt.hour == 1)[0].tolist()
    
    np.random.seed(seed)
    gap_rows = gap_days * rows_per_day
    
    # Dictionary lưu lại các index bị xóa để sau này tính RMSE, MAE
    missing_ground_truth = {col: [] for col in feature_columns}

    for col in feature_columns:
        col_idx = df_temp.columns.get_loc(col)
        
        # Chỉ giữ lại các vị trí bắt đầu mà khi cộng thêm gap_rows không vượt quá chiều dài data
        valid_starts = [pos for pos in daily_start_positions if pos + gap_rows <= len(df_temp)]
        
        available_starts = valid_starts.copy()
        
        for _ in range(num_gaps):
            if not available_starts:
                print(f"Cảnh báo: Không đủ khoảng trống để tạo đủ {num_gaps} gaps cho cột {col}")
                break
                
            # Chọn ngẫu nhiên 1 vị trí bắt đầu
            start_pos = np.random.choice(available_starts)
            end_pos = start_pos + gap_rows
            
            # Xóa dữ liệu (gán NaN)
            df_temp.iloc[start_pos:end_pos, col_idx] = np.nan
            
            # Lưu lại vị trí đã đục lỗ
            missing_ground_truth[col].extend(list(range(start_pos, end_pos)))
            
            # --- CƠ CHẾ CHỐNG ĐÈ (ANTI-OVERLAP) ---
            # Xóa bỏ các vị trí bắt đầu (start_pos) lân cận ra khỏi danh sách available_starts
            # Khoảng cách tối thiểu giữa 2 điểm bắt đầu phải lớn hơn chiều dài của gap
            available_starts = [pos for pos in available_starts if abs(pos - start_pos) > gap_rows]

    return df_temp, missing_ground_truth

# ================= TẠO 4 BỘ DATASET ĐỘC LẬP =================

# Giả sử 'df' là dataframe gốc của bạn
gap_scenarios = [1, 3, 5, 7]
missing_datasets = {}       # Chứa 4 dataframe đã bị đục lỗ
ground_truth_indices = {}   # Chứa vị trí các lỗ hổng để tính sai số sau này

for days in gap_scenarios:
    print(f"Đang tạo dataset cho kịch bản missing {days} ngày liên tục...")
    
    # Gọi hàm cho từng độ dài
    df_miss, truth_dict = generate_fixed_length_gaps(
        df_input=df, 
        gap_days=days, 
        num_gaps=10,   # Tùy chỉnh số lượng đoạn đứt gãy bạn muốn tạo
        seed=42,       # Giữ nguyên seed để kết quả random có thể tái lập được
        rows_per_day=8
    )
    
    # Lưu vào dictionary
    missing_datasets[f'gap_{days}d'] = df_miss
    ground_truth_indices[f'gap_{days}d'] = truth_dict
    
    total_nan = df_miss.isna().sum().sum()
    print(f"-> Hoàn tất! Tổng số NaN tạo ra: {total_nan}\n")

# Để truy xuất data sử dụng:
df_1_day = missing_datasets['gap_1d']
df_3_days = missing_datasets['gap_3d']
df_5_days = missing_datasets['gap_5d']
df_7_days = missing_datasets['gap_7d']

df_missing = df_7_days

In [ ]:
import missingno as msno
msno.matrix(df_missing)

In [ ]:
def split_by_hierarchical_correlation(df, low_thresh, high_thresh, time_col='TimeVN'):
    # Tách dữ liệu số để tính toán
    df_numeric = df.select_dtypes(include=[np.number])
    
    # 1. Ensemble Correlation
    corr = (df_numeric.corr(method='pearson').abs() + 
            df_numeric.corr(method='spearman').abs()) / 2
    
    # 2. Distance Matrix & Linkage (Dùng cho Dendrogram)
    dist_matrix = 1 - corr.fillna(0)
    dist_vec = squareform(dist_matrix, checks=False)
    linkage_matrix = hierarchy.ward(dist_vec)
    
    # 3. Tính Degree Centrality (Mức độ quan trọng trung bình của biến trong mạng lưới)
    global_scores = (corr.sum() - 1) / (len(corr) - 1)
    
    # 4. Phân tách theo 3 ngưỡng
    # Low: score < 0.3
    low_cols = global_scores[global_scores < low_thresh].index.tolist()
    # Medium: 0.3 <= score < 0.8
    med_cols = global_scores[(global_scores >= low_thresh) & (global_scores < high_thresh)].index.tolist()
    # High: score >= 0.8
    high_cols = global_scores[global_scores >= high_thresh].index.tolist()
    
    # 5. Trả kết quả (Kèm theo cột thời gian nếu có)
    time_list = [time_col] if time_col in df.columns else []
    
    df_low = df[time_list + low_cols]
    df_med = df[time_list + med_cols]
    df_high = df[time_list + high_cols]
    
    return df_low, df_med, df_high, global_scores, linkage_matrix

# --- Thực thi ---
df_low, df_med, df_high, scores, linkage = split_by_hierarchical_correlation(df_missing, low_thresh=0.6, high_thresh=0.75)

---
# **Models**

In [ ]:
import pandas as pd
import numpy as np
import gc
import os
from tqdm.auto import tqdm
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import warnings
warnings.filterwarnings("ignore")

# ==========================================
# 1. TẠO SEED (PRE-IMPUTATION: LINEAR VS LGBM)
# ==========================================
def get_seed_data(df_miss, method='lgbm'):
    """
    Tạo dữ liệu mồi. 
    'linear': Nội suy tuyến tính nhanh.
    'lgbm': Dùng LGBM dự đoán sơ bộ các điểm thiếu để làm feature cực tốt.
    """
    df_num = df_miss.select_dtypes(include=[np.number]).copy().astype('float32')

    if method == 'lgbm':
        # Bước đệm: Fill linear để có đầu vào cho LGBM Seed
        df_filled = df_num.interpolate(method='linear', limit_direction='both').ffill().bfill().fillna(0)
        cols = df_num.columns.tolist()
        
        for col in cols:
            mask = df_num[col].isna()
            if mask.any():
                features = [c for c in cols if c != col]
                X_train = df_filled.loc[~mask, features].values
                y_train = df_num.loc[~mask, col].values
                X_test = df_filled.loc[mask, features].values
                
                # Model seed nhanh
                seed_mdl = LGBMRegressor(n_estimators=50, n_jobs=-1, random_state=42, verbose=-1)
                seed_mdl.fit(X_train, y_train)
                df_filled.loc[mask, col] = seed_mdl.predict(X_test)
        return df_filled
    
    return df_num.fillna(0)

# ==========================================
# 2. MACHINE LEARNING MODELS DICTIONARY
# ==========================================
def get_ml_models():
    return {
        'LN': LinearRegression(n_jobs=-1),
        'Ridge': Ridge(alpha=1.0),
        'Lasso': Lasso(alpha=0.1),
        'KNN': KNeighborsRegressor(n_neighbors=5, n_jobs=-1),
        'DT': DecisionTreeRegressor(random_state=42),
        'SVR': SVR(kernel='rbf'), 
        'RF': RandomForestRegressor(n_estimators=50, n_jobs=-1, random_state=42),
        'GB': GradientBoostingRegressor(n_estimators=50, random_state=42),
        'Ada': AdaBoostRegressor(n_estimators=50, random_state=42),
        'XGB': XGBRegressor(n_estimators=50, n_jobs=-1, random_state=42, verbosity=0),
        'LGBM': LGBMRegressor(n_estimators=50, n_jobs=-1, random_state=42, verbose=-1)
    }

def run_ml_imputation(df_miss, df_seed, model_name):
    df_num = df_miss.select_dtypes(include=[np.number]).astype('float32')
    target_cols = df_num.columns.tolist()
    working = df_seed[target_cols].copy()
    
    models_dict = get_ml_models()
    model = models_dict[model_name]
    
    # One-vs-Rest: Duyệt qua từng cột để refine giá trị từ Seed
    pbar_targets = tqdm(target_cols, desc=f"⏳ Imputing {model_name:<5}", leave=False)
    for target in pbar_targets:
        m_idx = df_num[target].isna()
        if not m_idx.any(): continue
        
        pred_cols = [c for c in target_cols if c != target]
        
        # X lấy từ Seed chất lượng cao, y lấy từ giá trị thực (non-NaN)
        X_train = working.loc[~m_idx, pred_cols].values
        y_train = df_num.loc[~m_idx, target].values 
        X_test = working.loc[m_idx, pred_cols].values
        
        model.fit(X_train, y_train)
        working.loc[m_idx, target] = model.predict(X_test)
        
    return working

# ==========================================
# 3. METRICS EVALUATION
# ==========================================
def get_final_metrics(df_orig, df_imp, df_miss):
    stats = []
    common_cols = [c for c in df_miss.columns if c in df_orig.columns and c in df_imp.columns and c not in ['TimeVN', 'Time', 'Date']]
    for col in common_cols:
        mask = pd.isna(df_miss[col].values) & ~pd.isna(df_orig[col].values)
        yt, yp = df_orig[col].values[mask], df_imp[col].values[mask]
        valid = ~np.isnan(yp); yt, yp = yt[valid], yp[valid]
        if len(yt) < 2: continue
        
        rng = np.nanmax(df_orig[col].values) - np.nanmin(df_orig[col].values)
        rng = rng if rng > 0 else 1.0
        
        stats.append({
            "Station": col,
            "NSE": 1 - (np.sum((yt - yp)**2) / (np.sum((yt - np.mean(yt))**2) + 1e-9)),
            "R2": r2_score(yt, yp),
            "RMSE": np.sqrt(mean_squared_error(yt, yp)),
            "MAE": np.mean(np.abs(yt - yp)),
            "Sim": (1/len(yt)) * np.sum(1 / (1 + (np.abs(yp - yt) / rng)))
        })
    return pd.DataFrame(stats)

# ==========================================
# 4. EXECUTION PIPELINE
# ==========================================
groups_to_test = [('LOW', df_low), ('MED', df_med), ('HIGH', df_high)]
seed_methods = ['linear', 'lgbm']
models_to_test = list(get_ml_models().keys())

comparison_data = []

print("🚀 Khởi động One-vs-Rest Imputation (Seed: Linear vs LGBM)...\n")

pbar_levels = tqdm(groups_to_test, desc="📊 LEVEL", colour='magenta', position=0)
for level_name, df_group_missing in pbar_levels:
    
    pbar_seeds = tqdm(seed_methods, desc=f"🌟 SEED ({level_name})", colour='blue', position=1, leave=False)
    for SEED_METHOD in pbar_seeds:
        
        # Bước quan trọng: Tạo Seed mạnh
        df_seed_all = get_seed_data(df_group_missing, method=SEED_METHOD)
        
        pbar_models = tqdm(models_to_test, desc=f"🏆 MODEL ({SEED_METHOD})", colour='green', position=2, leave=False)
        for model_name in pbar_models:
            
            df_imp = run_ml_imputation(df_group_missing, df_seed_all, model_name)
            metrics_df = get_final_metrics(df, df_imp, df_group_missing)
            
            if not metrics_df.empty:
                s = metrics_df.mean(numeric_only=True)
                comparison_data.append({
                    'Level': level_name,
                    'Seed_Method': f"SEED_{SEED_METHOD.upper()}",
                    'Model': model_name,
                    'NSE (↑)': s['NSE'], 'R2 (↑)': s['R2'], 'Sim (↑)': s['Sim'],
                    'RMSE (↓)': s['RMSE'], 'MAE (↓)': s['MAE']
                })
            
            del df_imp; gc.collect()

# ==========================================
# 5. EXPORT
# ==========================================
final_df = pd.DataFrame(comparison_data)
final_df['Level'] = pd.Categorical(final_df['Level'], categories=['LOW', 'MED', 'HIGH'], ordered=True)
final_df = final_df.sort_values(by=['Level', 'Seed_Method', 'Model']).set_index(['Level', 'Seed_Method', 'Model'])

final_df.to_csv("ovr_lgbm_seed_results.csv")
print(f"\n✅ Hoàn tất! Dữ liệu lưu tại: ovr_lgbm_seed_results.csv")
display(final_df.style.format(precision=4).background_gradient(cmap='RdYlGn', subset=['NSE (↑)', 'R2 (↑)']))